# RNS LLM v0.14.2 — Full-RNS + Hybrid + Attention + PPL + Nsight

Исправленный воспроизводимый протокол для статьи:

- FP16 / FP32-reference / native INT8;
- full-RNS v0.7 и generalized full-RNS INT8/16/32;
- hybrid INT8-main + FP16/RNS correction;
- LUT `none / one / two / all` в обычных benchmark;
- fused QKV + `out_proj` внутри полного OPT Self-Attention;
- 4 CUDA streams;
- WikiText-2 PPL с фактическим CUDA-path;
- Nsight Systems и Nsight Compute в **article-essential режиме до 55 минут**;
- полный SQLite/SQL/JSON для Systems и `.ncu-rep`/CSV/JSON для Compute.

Для полного режима рекомендуется NVIDIA GPU с 16+ GB VRAM.

In [ ]:
# =========================
# Experiment configuration
# =========================
PAPER_MODE = True

RUN_PREFLIGHT = True
RUN_MATRIX = True
RUN_ATTENTION = True
RUN_PPL = True
RUN_NSYS = True
RUN_NCU = True
RUN_SUMMARY = True

MODEL_ID = "facebook/opt-2.7b" if PAPER_MODE else "facebook/opt-125m"
ATTENTION_SEQ = 128 if PAPER_MODE else 32
PPL_BLOCKS = 4 if PAPER_MODE else 1
PPL_TOKENS = 32768 if PAPER_MODE else 4096
CALIBRATION_TOKENS = 2048 if PAPER_MODE else 512

# Nsight defaults: only paper-essential representative profiles.
# Hard total limit is enforced by the project script.
NSIGHT_TOTAL_MINUTES = 55
NSIGHT_NCU_MODE = "essential"  # "essential" or optional exhaustive "full"
SKIP_EXISTING_NSIGHT = True

print({
    "PAPER_MODE": PAPER_MODE,
    "MODEL_ID": MODEL_ID,
    "RUN_MATRIX": RUN_MATRIX,
    "RUN_ATTENTION": RUN_ATTENTION,
    "RUN_PPL": RUN_PPL,
    "RUN_NSYS": RUN_NSYS,
    "RUN_NCU": RUN_NCU,
    "NSIGHT_TOTAL_MINUTES": NSIGHT_TOTAL_MINUTES,
    "NSIGHT_NCU_MODE": NSIGHT_NCU_MODE,
})

In [ ]:
# Upload and verify the v0.14.2 release ZIP
from google.colab import files
from pathlib import Path
import hashlib, zipfile, shutil, json

uploaded = files.upload()
zip_candidates = [Path(name) for name in uploaded if name.endswith(".zip") and "v014_2" in name.lower()]
if not zip_candidates:
    raise RuntimeError("Upload rns_llm_v014_2_full_hybrid_attention_project.zip")

release_zip = zip_candidates[0]
release_root = Path("/content/rns_llm_v014_2_full_hybrid_attention")
if release_root.exists():
    shutil.rmtree(release_root)
with zipfile.ZipFile(release_zip) as archive:
    archive.extractall("/content")
if not release_root.exists():
    matches = list(Path("/content").glob("rns_llm_v014_2_full_hybrid_attention"))
    if len(matches) != 1:
        raise RuntimeError(f"Cannot locate extracted release directory: {matches}")
    release_root = matches[0]

manifest = release_root / "MANIFEST_SHA256.txt"
if not manifest.is_file():
    raise RuntimeError("Release manifest is missing")
errors = []
for line in manifest.read_text().splitlines():
    if not line.strip():
        continue
    digest, relative = line.split("  ", 1)
    path = release_root / relative
    if not path.is_file():
        errors.append(f"missing: {relative}")
        continue
    actual = hashlib.sha256(path.read_bytes()).hexdigest()
    if actual != digest:
        errors.append(f"checksum mismatch: {relative}")
if errors:
    raise RuntimeError("Release verification failed:\n" + "\n".join(errors))
print("Release verified:", release_root)

In [ ]:
# Clone the pinned base repository and apply the v0.14.2 overlay
import subprocess, os, shutil
from pathlib import Path

repo = Path("/content/rns-llm")
if repo.exists():
    shutil.rmtree(repo)
base_commit = (release_root / "BASE_COMMIT").read_text().strip()
subprocess.run(["git", "clone", "https://github.com/Nek1tt/rns-llm.git", str(repo)], check=True)
subprocess.run(["git", "-C", str(repo), "checkout", base_commit], check=True)
subprocess.run(["bash", str(release_root / "apply_to_repo.sh"), str(repo)], check=True)
os.chdir(repo)
print("Repository:", repo)
print("Pinned commit:", subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip())

In [ ]:
# Install dependencies and compile all CUDA extensions
import subprocess, sys, os, torch

assert torch.cuda.is_available(), "Select a GPU runtime before continuing."
subprocess.run(["nvidia-smi"], check=False)
print("GPU:", torch.cuda.get_device_name(0))
print("PyTorch:", torch.__version__)
print("CUDA runtime:", torch.version.cuda)

subprocess.run([
    sys.executable, "-m", "pip", "install", "-q", "--upgrade",
    "pip", "setuptools<82", "wheel", "ninja",
], check=True)
subprocess.run([
    sys.executable, "-m", "pip", "install", "-q",
    "pandas>=2,<3", "matplotlib>=3.7",
    "transformers>=4.45,<5", "datasets>=2.19,<5",
    "accelerate>=0.33", "safetensors>=0.4", "huggingface_hub>=0.24",
], check=True)

env = os.environ.copy()
env["RNS_LLM_BUILD_CUDA"] = "1"
subprocess.run([
    sys.executable, "-m", "pip", "install", ".",
    "--no-build-isolation", "--no-deps", "--force-reinstall",
], check=True, env=env)

import rns_llm
print("rns_llm version:", rns_llm.__version__)
assert rns_llm.__version__ == "0.14.2"

In [ ]:
# Save exact run configuration
from pathlib import Path
import json, subprocess, torch

result_root = Path("results/v0.14.2")
report_root = Path("reports/v0.14.2")
result_root.mkdir(parents=True, exist_ok=True)
report_root.mkdir(parents=True, exist_ok=True)
config = {
    "version": "0.14.2",
    "paper_mode": PAPER_MODE,
    "model_id": MODEL_ID,
    "attention_seq": ATTENTION_SEQ,
    "ppl_blocks": PPL_BLOCKS,
    "ppl_tokens": PPL_TOKENS,
    "calibration_tokens": CALIBRATION_TOKENS,
    "run_preflight": RUN_PREFLIGHT,
    "run_matrix": RUN_MATRIX,
    "run_attention": RUN_ATTENTION,
    "run_ppl": RUN_PPL,
    "run_nsys": RUN_NSYS,
    "run_ncu": RUN_NCU,
    "nsight_total_minutes": NSIGHT_TOTAL_MINUTES,
    "nsight_ncu_mode": NSIGHT_NCU_MODE,
    "gpu": torch.cuda.get_device_name(0),
    "torch": torch.__version__,
    "cuda_runtime": torch.version.cuda,
    "git_commit": subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip(),
}
(result_root / "notebook_config_v0142.json").write_text(json.dumps(config, indent=2))
print(json.dumps(config, indent=2))

In [ ]:
# Mandatory GPU preflight: all architectures, q8/q16/q32, LUT equivalence,
# serial/parallel hybrid paths and finite-output checks.
import subprocess, sys
if RUN_PREFLIGHT:
    subprocess.run([
        sys.executable, "scripts/preflight_v014.py",
        "--output", "results/v0.14.2/preflight_v014.json",
    ], check=True)

In [ ]:
# Unified matrix benchmark
import subprocess, sys, json
from pathlib import Path
if RUN_MATRIX:
    if PAPER_MODE:
        cmd = [
            sys.executable, "benchmarks/benchmark_unified_matrix_v014.py",
            "--shape", "16x2560x2560",
            "--shape", "128x2560x2560",
            "--shape", "16x2560x10240",
            "--shape", "128x2560x10240",
            "--full-bits", "8", "16", "32",
            "--hybrid-bits", "8", "16", "32",
            "--moduli-policies", "dense_coprime", "large_primes", "school_small",
            "--lut-policies", "none", "one", "two", "all",
            "--protected", "3",
            "--warmup", "8", "--iterations", "30", "--repeats", "3",
            "--concurrency", "4",
            "--output-dir", "results/v0.14.2/matrix",
        ]
    else:
        cmd = [
            sys.executable, "benchmarks/benchmark_unified_matrix_v014.py",
            "--shape", "16x768x768", "--shape", "128x768x768",
            "--full-bits", "8", "16", "--hybrid-bits", "8", "16",
            "--moduli-policies", "dense_coprime", "school_small",
            "--lut-policies", "none", "two", "all",
            "--protected", "3", "--warmup", "4", "--iterations", "10",
            "--repeats", "1", "--concurrency", "4",
            "--output-dir", "results/v0.14.2/matrix",
        ]
    print(" ".join(cmd))
    subprocess.run(cmd, check=True)
    matrix_payload = json.loads(
        Path("results/v0.14.2/matrix/matrix_benchmark_v014.json").read_text()
    )
    if matrix_payload.get("errors"):
        raise RuntimeError(
            "Matrix benchmark contains failed variants:\n"
            + json.dumps(matrix_payload["errors"], indent=2)
        )

In [ ]:
# Complete OPT Self-Attention benchmark:
# fused QKV + native QK^T/mask/Softmax/AV + replaced out_proj.
import subprocess, sys, json
from pathlib import Path
if RUN_ATTENTION:
    cmd = [
        sys.executable, "benchmarks/benchmark_attention_v014.py",
        "--model", MODEL_ID, "--seq", str(ATTENTION_SEQ), "--protected", "3",
        "--full-bits", "8", "16", "32", "--hybrid-bits", "8", "16", "32",
        "--lut-policies", "none", "one", "two", "all",
        "--moduli-policy", "dense_coprime",
        "--warmup", "8" if PAPER_MODE else "3",
        "--iterations", "30" if PAPER_MODE else "8",
        "--concurrency", "4",
        "--output-dir", "results/v0.14.2/attention",
    ]
    print(" ".join(cmd))
    subprocess.run(cmd, check=True)
    attention_payload = json.loads(
        Path("results/v0.14.2/attention/attention_benchmark_v014.json").read_text()
    )
    if attention_payload.get("errors"):
        raise RuntimeError(
            "Attention benchmark contains failed variants:\n"
            + json.dumps(attention_payload["errors"], indent=2)
        )

In [ ]:
# WikiText-2 PPL using actual CUDA QKV/out projection paths.
# LUT is mathematically exact; PPL is run with the required two-table policy,
# while none/one/all are exhaustively compared in latency/accuracy benchmarks.
import subprocess, sys, json
from pathlib import Path
if RUN_PPL:
    cmd = [
        sys.executable, "scripts/evaluate_ppl_unified_v014.py",
        "--model", MODEL_ID,
        "--attention-blocks", str(PPL_BLOCKS),
        "--max-eval-tokens", str(PPL_TOKENS),
        "--calibration-tokens", str(CALIBRATION_TOKENS),
        "--dataset-split", "test", "--calibration-split", "validation",
        "--variants",
        "native_int8",
        "full_rns_int8_v07", "full_rns_int8", "full_rns_int16", "full_rns_int32",
        "hybrid_fp16", "hybrid_rns_q8", "hybrid_rns_q16", "hybrid_rns_q32",
        "--lut-policies", "two",
        "--moduli-policy", "dense_coprime",
        "--output-dir", "results/v0.14.2/ppl",
    ]
    print(" ".join(cmd))
    subprocess.run(cmd, check=True)
    ppl_payload = json.loads(
        Path("results/v0.14.2/ppl/ppl_unified_v014.json").read_text()
    )
    ppl_errors = [
        row for row in ppl_payload.get("results", [])
        if row.get("status") == "ERROR"
    ]
    if ppl_errors:
        raise RuntimeError(
            "PPL evaluation contains execution errors:\n"
            + json.dumps(ppl_errors, indent=2)
        )

In [ ]:
# Install/verify Nsight CLI tools. Existing valid tools are reused.
import subprocess, os
if RUN_NSYS or RUN_NCU:
    env = os.environ.copy()
    env["INSTALL_NCU"] = "1" if RUN_NCU else "0"
    subprocess.run(["bash", "scripts/install_nsight_colab_full.sh"], check=True, env=env)

In [ ]:
# Article-essential Nsight suite — hard total limit <= 55 minutes.
# NSYS: FP16, full-RNS INT8, hybrid RNS q16 on complete Attention.
# NCU: full-RNS and hybrid custom matrix kernels with essential sections.
import subprocess, sys
if RUN_NSYS or RUN_NCU:
    cmd = [
        sys.executable, "scripts/run_minimal_nsight_v014.py",
        "--output-root", "reports/v0.14.2",
        "--model", MODEL_ID,
        "--matrix-shape", "16x2560x2560" if PAPER_MODE else "16x768x768",
        "--attention-seq", str(min(ATTENTION_SEQ, 64)),
        "--total-minutes", str(NSIGHT_TOTAL_MINUTES),
        "--ncu-mode", NSIGHT_NCU_MODE,
        "--run-nsys" if RUN_NSYS else "--no-run-nsys",
        "--run-ncu" if RUN_NCU else "--no-run-ncu",
        "--skip-existing" if SKIP_EXISTING_NSIGHT else "--no-skip-existing",
    ]
    print(" ".join(cmd))
    subprocess.run(cmd, check=True)

In [ ]:
# Generate unified JSON/CSV/LaTeX/figure summaries
import subprocess, sys
if RUN_SUMMARY:
    subprocess.run([
        sys.executable, "scripts/summarize_v014.py",
        "--root", "results/v0.14.2",
        "--reports-root", "reports/v0.14.2",
        "--output-dir", "results/v0.14.2/summary",
    ], check=True)

In [ ]:
# Collect and download the complete result bundle
import subprocess
from pathlib import Path
from google.colab import files
bundle = Path("/content/rns_llm_v0142_results.zip")
if bundle.exists():
    bundle.unlink()
subprocess.run(["bash", "scripts/collect_v014_bundle.sh", ".", str(bundle)], check=True)
if not bundle.is_file() or bundle.stat().st_size == 0:
    raise RuntimeError("Result bundle was not created")
print("Bundle size:", bundle.stat().st_size, "bytes")
files.download(str(bundle))

## Что прислать обратно

Пришли `rns_llm_v0142_results.zip`. Для статьи особенно важны:

- `results/v0.14.2/matrix/`;
- `results/v0.14.2/attention/`;
- `results/v0.14.2/ppl/`;
- `results/v0.14.2/summary/`;
- `reports/v0.14.2/nsys/*.nsys-rep`, `*.sqlite`, `*_queries.sql`, `*_sql_summary.json`;
- `reports/v0.14.2/ncu/*.ncu-rep`, `*_raw.csv`, `*_raw_full.json`, `*_details_full.json`.